<a href="https://colab.research.google.com/github/Diego-Rodriguez16/mini-robots-2026-II/blob/main/3_Algoritmos_geneticos/Taller_3_Algoritmos_Geneticos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos Genéticos — Ejercicios 1, 2 y 4 (versión simple)

Tres algoritmos genéticos (AG) escritos desde cero con NumPy, lo más sencillos posible.
Pasos de todos los AG: **población → aptitud → selección → cruce → mutación → repetir**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import data
from skimage.transform import resize

rng = np.random.default_rng(0)

## Ejercicio 1 — Maximizar $f(x)=x\,\sin(10\pi x)+1$ en $[0,1]$

- **Cromosoma:** 20 bits que se convierten a un número real entre 0 y 1.
- **Aptitud:** el valor de $f(x)$ (se busca el mayor).
- **Selección:** torneo entre 2 individuos.
- **Cruce:** un punto. **Mutación:** cambiar bits con probabilidad baja. **Elitismo:** se conserva el mejor.

In [ ]:
def f(x):
    return x * np.sin(10 * np.pi * x) + 1

def decodificar(pob):
    pesos = 2 ** np.arange(19, -1, -1)          # valor de cada bit
    return (pob @ pesos) / (2**20 - 1)          # número entre 0 y 1

N, BITS, GENERACIONES = 100, 20, 100
pob = rng.integers(0, 2, size=(N, BITS))        # población inicial aleatoria
mejores, promedios = [], []

for g in range(GENERACIONES):
    x = decodificar(pob)
    apt = f(x)
    mejores.append(apt.max()); promedios.append(apt.mean())
    mejor = pob[apt.argmax()].copy()

    # selección por torneo de 2
    a = rng.integers(0, N, N); b = rng.integers(0, N, N)
    padres = pob[np.where(apt[a] > apt[b], a, b)]

    # cruce de un punto entre parejas
    hijos = padres.copy()
    for i in range(0, N, 2):
        c = rng.integers(1, BITS)
        hijos[i, c:], hijos[i+1, c:] = padres[i+1, c:], padres[i, c:]

    # mutación
    mut = rng.random(hijos.shape) < 0.01
    hijos = np.where(mut, 1 - hijos, hijos)

    hijos[0] = mejor                             # elitismo
    pob = hijos

x = decodificar(pob); apt = f(x)
print(f"Mejor x = {x[apt.argmax()]:.5f}   f(x) = {apt.max():.5f}")

In [ ]:
xx = np.linspace(0, 1, 1000)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(xx, f(xx), label="f(x)")
ax[0].scatter(x, apt, color="red", s=15, label="población final")
ax[0].set_title("Función y población final"); ax[0].legend()
ax[1].plot(mejores, label="mejor"); ax[1].plot(promedios, label="promedio")
ax[1].set_title("Convergencia"); ax[1].set_xlabel("Generación"); ax[1].legend()
plt.tight_layout(); plt.show()
print("Máximo real (búsqueda directa):", round(f(xx).max(), 5), "en x =", round(xx[f(xx).argmax()], 5))

**Análisis.** La población empieza dispersa y termina agrupada en el máximo global (x ≈ 0.85). El resultado coincide con el máximo calculado por búsqueda directa.

## Ejercicio 2 — Verdadera democracia

Se reparten **50 entidades** (peso político de 1 a 100) entre **5 partidos** en proporción a sus curules.

- **Cromosoma:** vector de 50 números (0–4); el gen $i$ dice a qué partido va la entidad $i$.
- **Poder objetivo del partido $j$:** poder total × curules$_j$ / 50.
- **Aptitud (se minimiza):** $\sum_j |\text{poder asignado}_j - \text{objetivo}_j|$.
- **Selección:** torneo de 2. **Cruce:** uniforme. **Mutación:** cambiar el partido de una entidad. **Elitismo:** el mejor.

In [ ]:
# Datos aleatorios
cortes = np.sort(rng.choice(np.arange(1, 50), size=4, replace=False))
curules = np.diff([0, *cortes, 50])              # 5 partidos, 50 curules, mínimo 1 cada uno
pesos = rng.integers(1, 101, size=50)            # peso de cada entidad
objetivo = pesos.sum() * curules / 50
partidos = ["A", "B", "C", "D", "E"]
print("Curules:", curules, "| Poder total:", pesos.sum())

def error(ind):
    poder = np.array([pesos[ind == j].sum() for j in range(5)])
    return np.abs(poder - objetivo).sum()

N, GENERACIONES = 100, 300
pob = rng.integers(0, 5, size=(N, 50))
hist = []

for g in range(GENERACIONES):
    err = np.array([error(ind) for ind in pob])
    hist.append(err.min())
    mejor = pob[err.argmin()].copy()

    a = rng.integers(0, N, N); b = rng.integers(0, N, N)
    padres = pob[np.where(err[a] < err[b], a, b)]

    # cruce uniforme: cada gen viene de uno de los dos padres
    mascara = rng.random((N, 50)) < 0.5
    hijos = np.where(mascara, padres, np.roll(padres, 1, axis=0))

    # mutación: reasignar al azar
    mut = rng.random(hijos.shape) < 0.02
    hijos = np.where(mut, rng.integers(0, 5, hijos.shape), hijos)

    hijos[0] = mejor
    pob = hijos

err = np.array([error(ind) for ind in pob])
mejor = pob[err.argmin()]
print("Error final:", err.min())

In [ ]:
# Matriz de poder 50 x 5: el peso de la entidad va en la columna del partido que la recibe
matriz = np.zeros((50, 5), dtype=int)
matriz[np.arange(50), mejor] = pesos
matriz_poder = pd.DataFrame(matriz, index=[f"Entidad {i+1}" for i in range(50)],
                            columns=[f"Partido {p}" for p in partidos])

resumen = pd.DataFrame({"Curules": curules,
                        "Poder objetivo": objetivo.round(1),
                        "Poder asignado": matriz.sum(axis=0)},
                       index=matriz_poder.columns)
display(resumen)
display(matriz_poder.head(10))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(5)
ax[0].bar(x - 0.2, objetivo, 0.4, label="Objetivo")
ax[0].bar(x + 0.2, matriz.sum(axis=0), 0.4, label="Asignado")
ax[0].set_xticks(x); ax[0].set_xticklabels(partidos); ax[0].legend()
ax[0].set_title("Poder objetivo vs. asignado")
ax[1].plot(hist); ax[1].set_title("Convergencia (error)"); ax[1].set_xlabel("Generación")
plt.tight_layout(); plt.show()

**Análisis.** El poder asignado a cada partido queda casi igual al que le corresponde por sus curules. El error no llega exactamente a 0 porque las entidades no se pueden partir (los pesos son enteros).

## Ejercicio 4 — Evolución de una imagen

Se crean **50 matrices de 120×180×3** (RGB) con enteros aleatorios de 0 a 255. La **aptitud** es el error cuadrático medio (MSE) frente a una imagen objetivo (se minimiza).

- **Selección:** torneo de 2 por cada padre.
- **Cruce:** para cada píxel, el hijo toma el valor del padre que tenga menos error en ese píxel (así se aprovecha que el error total es la suma de los errores por píxel; con un cruce totalmente ciego, 64 800 genes harían la evolución muy lenta).
- **Mutación:** a algunos valores se les suma ruido pequeño.
- **Elitismo:** el mejor pasa intacto.

In [ ]:
objetivo_img = (resize(data.astronaut(), (120, 180), anti_aliasing=True) * 255).round()

N = 50
pob = rng.integers(0, 256, size=(N, 120, 180, 3)).astype(float)   # 50 matrices aleatorias

# Población inicial como imágenes RGB (se muestran 10 de las 50)
fig, axs = plt.subplots(2, 5, figsize=(12, 5))
for k, ax in enumerate(axs.ravel()):
    ax.imshow(pob[k].astype(np.uint8)); ax.axis("off"); ax.set_title(f"Individuo {k+1}")
plt.suptitle("Población inicial (ruido aleatorio)"); plt.show()

In [ ]:
GENERACIONES = 300
mostrar = [0, 5, 20, 100, GENERACIONES]
capturas, hist = {}, []

for g in range(GENERACIONES + 1):
    sq = (pob - objetivo_img) ** 2
    err = sq.mean(axis=(1, 2, 3))                 # MSE de cada individuo
    i = err.argmin()
    hist.append(err[i])
    if g in mostrar:
        capturas[g] = pob[i].astype(np.uint8)
    if g % 50 == 0:
        print(f"Generación {g:4d} | mejor MSE = {err[i]:.1f}")
    if g == GENERACIONES:
        break

    # selección por torneo de 2 (dos padres por hijo)
    def torneo():
        a = rng.integers(0, N, N); b = rng.integers(0, N, N)
        return np.where(err[a] < err[b], a, b)
    pa, pb = torneo(), torneo()

    # cruce: en cada píxel gana el padre con menor error local
    local = sq.sum(axis=3)
    a_gana = local[pa] <= local[pb]
    hijos = np.where(a_gana[..., None], pob[pa], pob[pb])

    # mutación: ruido a un 3 % de los valores (disminuye con las generaciones)
    sigma = 30 * (1 - g / GENERACIONES) + 2
    mut = rng.random(hijos.shape) < 0.03
    hijos = np.clip(hijos + mut * rng.normal(0, sigma, hijos.shape), 0, 255).round()

    hijos[0] = pob[i]                             # elitismo
    pob = hijos

In [ ]:
fig, axs = plt.subplots(1, len(mostrar) + 1, figsize=(16, 3))
for ax, g in zip(axs, mostrar):
    ax.imshow(capturas[g]); ax.set_title(f"Gen {g}\nMSE={hist[g]:.0f}"); ax.axis("off")
axs[-1].imshow(objetivo_img.astype(np.uint8)); axs[-1].set_title("OBJETIVO"); axs[-1].axis("off")
plt.show()

plt.figure(figsize=(6, 3.5))
plt.semilogy(np.array(hist) + 1e-3); plt.title("Convergencia (MSE)"); plt.xlabel("Generación")
plt.show()

**Análisis.** En pocas generaciones el ruido da paso a las formas generales de la imagen y después se van refinando los detalles. El MSE baja muy rápido al principio y más despacio al final, como es típico en un AG.